# Market Foresight Before Trading

This notebook runs the Agentic Finance launch demo offline. A finance agent checks a labor-market thesis with recorded PolyBridge-style probabilities before recording a simulated SPY paper trade through SimBroker.

This is research/demo software, not financial advice. It uses no brokerage account, no API key, and no real trading. The recorded replay also needs no extra packages — stock Python 3.9+ is enough.

Prefer the terminal? The recorded demo is one command:

```bash
git clone https://github.com/crowdvector/polybridge-cookbooks.git
cd polybridge-cookbooks/agentic-finance
bash demo.sh
```

## Flow

1. Load `labor-resilience-jul2026`.
2. Run the Evidence Gate from the recorded replay.
3. Write the decision memo and audit log.
4. Record a SimBroker simulated paper fill after explicit notebook confirmation.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/crowdvector/polybridge-cookbooks.git"

if Path("/content").exists():
    repo = Path("/content/polybridge-cookbooks")
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    base = repo / "agentic-finance"
    os.chdir(base)
else:
    base = Path("agentic-finance") if Path("agentic-finance").exists() else Path(".")

base = base.resolve()
sys.path.insert(0, str(base))
print("Cookbook folder:", base)

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

tier1 = load_module("tier1_evidence_gate", base / "tier1_evidence_gate.py")
tier3 = load_module("tier3_paper_trader", base / "tier3_paper_trader.py")

thesis_id = "labor-resilience-jul2026"
replay_path = base / "examples" / "recorded_run_2026-07-04.json"
theses_path = base / "examples" / "sample_theses.json"
outputs = base / "outputs" / "notebook-demo"
outputs.mkdir(parents=True, exist_ok=True)

In [ ]:
tier1_result = tier1.run_replay(
    thesis_id=thesis_id,
    replay_path=replay_path,
    theses_path=theses_path,
    output_dir=outputs,
    base_dir=base,
)

decision = tier1_result["multi_leg_decision"]
print("VERDICT:", decision.verdict)
print("weighted_support:", decision.weighted_support)
for label, path in tier1_result["paths"].items():
    print(f"{label}: {path}")

In [ ]:
def preview_text(path, max_chars=1200):
    text = Path(path).read_text(encoding="utf-8")
    print(text[:max_chars])

print("decision-memo.md")
preview_text(tier1_result["paths"]["decision_memo"])

print("\nevidence-packet.json")
packet = json.loads(Path(tier1_result["paths"]["evidence_packet"]).read_text(encoding="utf-8"))
print(json.dumps(packet, indent=2)[:1200])

## Human Approval

Running the next cell is the human approval step for the local SimBroker demo. Change `confirmation` to `"n"` to decline; a decline records `human_declined` and writes no simulated fill.

In [ ]:
confirmation = "y"  # Change to "n" to decline. Running this cell is the human approval step.

tier3_result = tier3.run_paper_trader(
    thesis_id=thesis_id,
    replay_path=replay_path,
    theses_path=theses_path,
    output_dir=outputs,
    confirmation=confirmation,
    base_dir=base,
)

print("VERDICT:", tier3_result["multi_leg_decision"].verdict)
print("BROKER:", tier3_result["broker_preview"]["broker_name"])
print("SYMBOL:", tier3_result["broker_preview"]["symbol"])
print("SIDE:", tier3_result["broker_preview"]["side"].upper())
print("NOTIONAL:", tier3_result["broker_preview"]["notional_usd"])
if tier3_result["broker_event"] == "simulated_fill_recorded":
    print("simulated_order_id:", tier3_result["broker_result"]["order_id"])
    print("paper_portfolio:", tier3_result["paths"]["paper_portfolio"])
else:
    print("human_declined: no simulated fill recorded")
print("audit_log:", tier3_result["paths"]["decisions_log"])

In [ ]:
if tier3_result["broker_event"] == "simulated_fill_recorded":
    print("paper_portfolio.jsonl")
    preview_text(tier3_result["paths"]["paper_portfolio"])
    print()

print("decisions.jsonl")
preview_text(tier3_result["paths"]["decisions_log"], max_chars=1600)